In [ ]:
import os
from pathlib import Path
import sys

if Path.cwd().name == "notebooks":
  os.chdir(Path.cwd().parent.parent)

print("Intérprete:", sys.executable)
print("Carpeta de trabajo:", Path.cwd())

for lib in ["numpy", "pandas", "matplotlib", "sklearn", "openpyxl"]:
  try:
    print(f" {lib:12}", __import__(lib).__version__)
  except ImportError:
    print(f" {lib:12} NO DISPONIBLE en este kernel")

# Fase 1: Definición del Problema, Entorno Reproducible y Validación del Dataset
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Integrantes:** Madeleine Aguilera, Victoria Álvarez, Ignacio Peña y Raúl Hormazábal

---

## 1. Situación Problematizadora y Pregunta Analizable
La gestión y fiscalización ambiental en Chile requiere analizar las declaraciones anuales de generación de residuos peligrosos registradas en el **Sistema de Declaración y Seguimiento de Residuos Peligrosos (SIDREP)** del **Registro de Emisiones y Transferencias de Contaminantes (RETC)**. Sin embargo, al consolidar la base maestra trianual **2023–2025** (`Base_Maestra_2023_2025.xlsx`, con 118.280 registros y 21 variables), emergen problemas reales de calidad de datos producto de cambios de formato interanual: **118 registros duplicados exactos**, **19 etiquetas regionales distintas para las 16 regiones de Chile**, **812 combinaciones heterogéneas en la glosa de peligrosidad** (por diferencias de mayúsculas, tildes y separadores `##` vs `;` en 2025), **pérdida del separador decimal en las coordenadas geográficas de 2024** y **valores faltantes en `id_lista_a` (4,03%), coordenadas y masa declarada (`cantidad_kg`)**.

- **Pregunta analizable del proyecto:** *¿Cómo se distribuye el volumen declarado de residuos peligrosos (`cantidad_kg`) y su perfil de peligrosidad según el rubro industrial, el estado físico de la materia y la ubicación regional en Chile durante el período 2023–2025?*
- **Objetivo General:** Implementar un flujo de trabajo reproducible y trazable en Python para explorar, depurar, normalizar, transformar y validar la base maestra de declaraciones de residuos peligrosos en Chile (2023–2025).
- **Objetivos Específicos (Fases 1 y 2):**
  1. Configurar y verificar un entorno computacional aislado (`.venv`), conectado al kernel del proyecto y versionado en Git/GitHub con higiene de notebooks (`nbstripout`).
  2. Validar que el conjunto de datos original (`F1/data/raw/Base_Maestra_2023_2025.xlsx`) cumple los requisitos multivariados y de volumen del curso sin alterar el archivo crudo.
  3. Diseñar en la Fase 2 un pipeline modular (`F2/src/preprocesamiento.py`) de deduplicación, estandarización interanual de regiones y peligrosidades, rescate de escala geográfica, imputación por grupo (`region` e `id_rubro`) y escalamiento robusto (`RobustScaler`).

In [ ]:
# Celda 3: Carga controlada del dataset original, diccionario de variables y validación Fase 1
import pandas as pd
from pathlib import Path
from IPython.display import display

RUTA_RAW = Path("F1/data/raw/Base_Maestra_2023_2025.xlsx")

# 1. Carga con manejo de excepciones (buena práctica de software científico)
try:
    if not RUTA_RAW.exists():
        raise FileNotFoundError(f"No se encontró el archivo en la ruta relativa: {RUTA_RAW}")
    df_raw = pd.read_excel(RUTA_RAW)
    print(f"[OK] Dataset cargado correctamente desde: {RUTA_RAW}")
    print(f"Dimensiones originales: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
except Exception as e:
    print(f"[ERROR] Fallo al cargar el dataset: {e}")
    raise

# 2. Diccionario de las 21 variables definido por el Grupo 3
DICCIONARIO_VARIABLES = {
    "año":                ("fecha",             "Año calendario de la declaración RETC (2023, 2024, 2025)"),
    "id_vu":              ("identificador",     "Identificador numérico del establecimiento en Ventanilla Única"),
    "rut_razon_social":   ("alta cardinalidad", "RUT tributario de la organización declarante (6.779 empresas)"),
    "id_ciiu4":           ("alta cardinalidad", "Código CIIU4 de actividad económica a 4 dígitos (546 códigos)"),
    "ciiu4":              ("texto libre",       "Glosa descriptiva de la actividad económica según CIIU4"),
    "id_ciiu6":           ("alta cardinalidad", "Código CIIU6 de actividad económica a 6 dígitos (388 códigos)"),
    "ciiu6":              ("texto libre",       "Glosa descriptiva detallada de la actividad económica según CIIU6"),
    "id_rubro":           ("discreta",          "Código entero del rubro económico RETC (0 a 20)"),
    "rubro":              ("nominal",           "Categoría de rubro industrial o sector productivo (19 rubros)"),
    "codigo_territorial": ("discreta",          "Código Único Territorial (CUT) comunal entero (380 códigos)"),
    "comuna":             ("alta cardinalidad", "Nombre de la comuna del establecimiento generador (344 comunas)"),
    "provincia":          ("nominal",           "Nombre de la provincia administrativa (58 provincias)"),
    "region":             ("nominal",           "Región administrativa de Chile (16 regiones; 19 etiquetas en crudo)"),
    "latitud":            ("continua",          "Latitud geográfica del establecimiento en grados decimales"),
    "longitud":           ("continua",          "Longitud geográfica del establecimiento en grados decimales"),
    "cantidad_kg":        ("continua",          "Masa anual declarada de residuo peligroso en kilogramos (kg)"),
    "id_peligrosidad":    ("ordinal",           "Códigos de característica de peligrosidad D.S. N° 148/2003"),
    "peligrosidad":       ("texto libre",       "Glosa de característica(s) de peligrosidad declaradas"),
    "id_lista_a":         ("nominal",           "Código de clasificación Lista A del D.S. N° 148/2003 MINSAL"),
    "id_estado_materia":  ("ordinal",           "Código de estado físico de agregación (1=sólido, 2=líquido, 3=gaseoso, 4=lodo)"),
    "estado_materia":     ("nominal",           "Estado físico del residuo (sólido, líquido, lodo, gaseoso)"),
}

# 3. Construcción del perfil técnico de las variables
perfil_f1 = pd.DataFrame([
    {
        "variable": col,
        "dtype_pandas": str(df_raw[col].dtype),
        "rol_analitico": DICCIONARIO_VARIABLES[col][0],
        "valores_unicos": int(df_raw[col].nunique(dropna=True)),
        "nulos_n": int(df_raw[col].isna().sum()),
        "nulos_pct": round(df_raw[col].isna().mean() * 100, 2),
        "descripcion": DICCIONARIO_VARIABLES[col][1],
    }
    for col in df_raw.columns
])

display(perfil_f1)

# 4. Verificación formal de los umbrales mínimos del curso mediante assert
roles_presentes = set(perfil_f1["rol_analitico"]) - {"identificador"}
max_pct_nulos = perfil_f1["nulos_pct"].max()
duplicados_n = int(df_raw.duplicated().sum())

assert df_raw.shape[0] >= 2000, "El dataset debe tener al menos 2.000 filas"
assert df_raw.shape[1] >= 12, "El dataset debe tener al menos 12 columnas"
assert len(roles_presentes) >= 7, "Debe cubrir múltiples roles analíticos"
assert 1.0 <= max_pct_nulos <= 60.0, "El porcentaje máximo de nulos debe estar entre 1% y 60%"

print("\nResumen de roles analíticos presentes:")
print(perfil_f1["rol_analitico"].value_counts().to_string())
print(f"\n[OK] Todos los requisitos verificados. Filas: {df_raw.shape[0]} | Duplicados exactos: {duplicados_n} | Máximo de nulos: {max_pct_nulos}% en id_lista_a.")

# 5. Guardar reporte Markdown de validación en F1/docs/informe_dataset.md
ruta_informe = Path("F1/docs/informe_dataset.md")
lineas_md = [
    "# Informe de Validación del Dataset (Fase 1) — Grupo 3 MCDI500",
    f"- **Archivo evaluado:** `{RUTA_RAW}`",
    f"- **Dimensiones:** {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas",
    f"- **Filas duplicadas exactas:** {duplicados_n}",
    f"- **Porcentaje máximo de valores faltantes:** {max_pct_nulos}% (`id_lista_a`: 4.764 nulos)",
    "",
    "## Diccionario y Perfil de Variables",
    "| " + " | ".join(perfil_f1.columns) + " |",
    "| " + " | ".join(["---"] * len(perfil_f1.columns)) + " |",
]
for fila in perfil_f1.values:
    lineas_md.append("| " + " | ".join(str(v) for v in fila) + " |")

ruta_informe.write_text("\n".join(lineas_md), encoding="utf-8")
print(f"[OK] Reporte exportado en: {ruta_informe}")